# Preprocessing
what all we will do

- Combine ticket subject + ticket description into one column to have both subject for clean topic of ticket and description will add context to it. Model will require only one column then

- Remove placeholders like {product_purchased} as they will add noise and confuse the model

- lowercase emntire text

- strip extra white spaces for better tockenization

- encode ticket type like "technical issue" into 0 and "biilong inquiry" to 1 as the model works with numbers, not strings

- Encode ticket priority to numbers the same way as ticket tye

- save the clean dataframe

In [67]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import re  ##regex to remobe the place holders

In [68]:
df = pd.read_csv('../Dataset/sample_tickets_dataset.csv')
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [69]:
## removing unnessary columns
df = df.drop(columns=['Ticket ID','Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender',
               'Date of Purchase','First Response Time','Time to Resolution','Resolution', 
               "Customer Satisfaction Rating"],errors='ignore')

In [70]:
print(df.shape)
print(df.dtypes)


(8469, 7)
Product Purchased     str
Ticket Type           str
Ticket Subject        str
Ticket Description    str
Ticket Status         str
Ticket Priority       str
Ticket Channel        str
dtype: object


In [71]:
## combining subject and description
df['combine_text'] = df["Ticket Subject"] + " " + df["Ticket Description"]

df['combine_text'].head()

0    Product setup I'm having an issue with the {pr...
1    Peripheral compatibility I'm having an issue w...
2    Network problem I'm facing a problem with my {...
3    Account access I'm having an issue with the {p...
4    Data loss I'm having an issue with the {produc...
Name: combine_text, dtype: str

In [72]:
## removing the placeholders using regex

df['combine_text'] = df['combine_text'].apply(lambda x: re.sub(r'\{.*?\}',"",x))

## this will replace anything within {} with "" i.e nothing

In [73]:
## changing all the text to lower and removing unneccessary spaces
df['combine_text'] = df['combine_text'].str.lower()

In [74]:
df['combine_text'] = df['combine_text'].str.strip()

In [75]:
df['combine_text'].head()

0    product setup i'm having an issue with the . p...
1    peripheral compatibility i'm having an issue w...
2    network problem i'm facing a problem with my ....
3    account access i'm having an issue with the . ...
4    data loss i'm having an issue with the . pleas...
Name: combine_text, dtype: str

one issue noticed here in the output:
there is a stray perid and extra space left behind where he had the place holder.
regex removed the placeholder, but the surrounding spaces and punctuations are still left behind

so we again apply regex and us it to colapse multple spaces into one to avoid stray period

In [76]:
df['combine_text'] = df['combine_text'].apply(lambda x: re.sub(r'\s+',' ',x).strip())
df['combine_text'].head()

0    product setup i'm having an issue with the . p...
1    peripheral compatibility i'm having an issue w...
2    network problem i'm facing a problem with my ....
3    account access i'm having an issue with the . ...
4    data loss i'm having an issue with the . pleas...
Name: combine_text, dtype: str

In [77]:
## Label encoding ticket type and ticket prioirty

# encoding ticket type

le_type = LabelEncoder()
df['ticket_type_encoded'] = le_type.fit_transform(df['Ticket Type'])

print(dict(zip(le_type.classes_, le_type.transform(le_type.classes_))))

{'Billing inquiry': np.int64(0), 'Cancellation request': np.int64(1), 'Product inquiry': np.int64(2), 'Refund request': np.int64(3), 'Technical issue': np.int64(4)}


In [78]:
## encoding ticket priority

le_priority = LabelEncoder()
df['ticket_priority_encoded'] = le_priority.fit_transform(df['Ticket Priority'])

print(dict(zip(le_priority.classes_, le_priority.transform(le_priority.classes_))))

{'Critical': np.int64(0), 'High': np.int64(1), 'Low': np.int64(2), 'Medium': np.int64(3)}


In [79]:
print(df[['Ticket Type', 'ticket_type_encoded', 'Ticket Priority', 'ticket_priority_encoded']].head(10))

            Ticket Type  ...  ticket_priority_encoded
0       Technical issue  ...                        0
1       Technical issue  ...                        0
2       Technical issue  ...                        2
3       Billing inquiry  ...                        2
4       Billing inquiry  ...                        2
5  Cancellation request  ...                        2
6       Product inquiry  ...                        0
7        Refund request  ...                        0
8       Technical issue  ...                        2
9        Refund request  ...                        0

[10 rows x 4 columns]


In [80]:
## saving the encoders

import joblib
joblib.dump(le_type , '../models/le_type.pkl')
joblib.dump(le_priority, "../models/le_priority.pkl")

print("encoders saved")

encoders saved


In [81]:
df.head()

,Product Purchased,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Ticket Priority,Ticket Channel,combine_text,ticket_type_encoded,ticket_priority_encoded
0,GoPro Hero,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,Critical,Social media,product setup i'm having an issue with the . p...,4,0
1,LG Smart TV,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,Critical,Chat,peripheral compatibility i'm having an issue w...,4,0
2,Dell XPS,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Low,Social media,network problem i'm facing a problem with my ....,4,2
3,Microsoft Office,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Low,Social media,account access i'm having an issue with the . ...,0,2
4,Autodesk AutoCAD,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,Low,Email,data loss i'm having an issue with the . pleas...,0,2


In [82]:
## dropping the columns we do not require any more

df = df.drop(columns=['Ticket Channel','Ticket Description'])
df.head(3)

,Product Purchased,Ticket Type,Ticket Subject,Ticket Status,Ticket Priority,combine_text,ticket_type_encoded,ticket_priority_encoded
0,GoPro Hero,Technical issue,Product setup,Pending Customer Response,Critical,product setup i'm having an issue with the . p...,4,0
1,LG Smart TV,Technical issue,Peripheral compatibility,Pending Customer Response,Critical,peripheral compatibility i'm having an issue w...,4,0
2,Dell XPS,Technical issue,Network problem,Closed,Low,network problem i'm facing a problem with my ....,4,2


In [83]:
## we kept the rest of the columns for following reasons:

# combine_text — model input
# ticket_type_encoded — model target 1
# ticket_priority_encoded — model target 2
# Ticket Type — for display in Streamlit
# Ticket Priority — for display in Streamlit
# Product Purchased — useful context to display
# Ticket Subject — useful for display in results

In [84]:
## saving the new dataset

df.to_csv('../Dataset/cleaned_tickets.csv',index=False)
print("dataset saved. Shape:", df.shape)
print("columns: ", df.columns.tolist())

dataset saved. Shape: (8469, 8)
columns:  ['Product Purchased', 'Ticket Type', 'Ticket Subject', 'Ticket Status', 'Ticket Priority', 'combine_text', 'ticket_type_encoded', 'ticket_priority_encoded']
